In [1]:
import sys
sys.path.append('../')

import pandas as pd
from functools import partial
from util.polymarket_client import PolymarketAPIClient
from trading_rules import entries as e, exits as ex, engine
from trading_rules.metrics import Metrics
from util.data_processor import TickDataIntervalEnum

In [ ]:
# FETCH THE DATA STEP AND PROCESS IT
MARKET_SLUG = 'will-jesus-christ-return-in-2025'
client = PolymarketAPIClient()
market = client.get_price_history_by_outcome(MARKET_SLUG, desired_outcome="No", interval=TickDataIntervalEnum.FIVE_MINUTES)
market["symbol"] = MARKET_SLUG
resampled_df = market.resample("10min").last()

Will Jesus Christ return in 2025?
Not Will Jesus Christ return in 2025?
requested start and end: 2024-12-31 12:00:00+00:00 2025-12-31 12:00:00+00:00


In [ ]:
resampled_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 41123 entries, 2025-03-20 22:10:00+00:00 to 2025-12-31 11:50:00+00:00
Freq: 10min
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   timestamp   41100 non-null  float64
 1   open        41100 non-null  float64
 2   high        41100 non-null  float64
 3   low         41100 non-null  float64
 4   close       41100 non-null  float64
 5   volume      41100 non-null  float64
 6   outcome_id  41100 non-null  object 
 7   symbol      41100 non-null  object 
dtypes: float64(6), object(2)
memory usage: 2.8+ MB


In [ ]:
# RUN THE SIMULATED BACKTEST

entries_partials = [
    partial(e.mean_reversion, data=resampled_df, window=10)
]
entries_series = [entry(data=resampled_df) for entry in entries_partials]
# AND all the entry rules
entries = pd.concat(entries_series, axis=1).all(axis=1)
exit_rules = [ex.time_exit(max_bars=144)]

result = engine.backtest(
    data=resampled_df,
    entries_series=entries,
    exits_list=exit_rules,
    initial_cash=10000,
)

result


<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.fl

{'equity': timestamp_formatted
 2025-03-20 22:10:00+00:00    10000.000000
 2025-03-20 22:20:00+00:00    10000.000000
 2025-03-20 22:30:00+00:00    10000.000000
 2025-03-20 22:40:00+00:00    10000.000000
 2025-03-20 22:50:00+00:00    10000.000000
                                  ...     
 2025-12-31 11:10:00+00:00    10384.415584
 2025-12-31 11:20:00+00:00    10384.415584
 2025-12-31 11:30:00+00:00    10384.415584
 2025-12-31 11:40:00+00:00    10384.415584
 2025-12-31 11:50:00+00:00    10384.415584
 Freq: 10min, Length: 41123, dtype: float64,
 'trades': []}

In [ ]:
# MEASURE PERFORMANCE USING METRICS
metrics = Metrics()